# FGNN2 strict scores – Colab, no DGL

DGL's download site is unreachable, so this notebook uses `fgnn2_torch.py`: the same FGNN2 math in plain
PyTorch (Colab already has PyTorch). It loads the same weight files from Google Drive.

**Runtime → Change runtime type → CPU** is enough.

**Correctness check (4b):** the *official* rows must match our earlier numbers
(pretrained: hard 0.780 / full 0.616, Exp B: hard 0.821 / full 0.676).

In [ ]:
# 1. setup: Drive, dataset, pretrained weights, ABC (no DGL, no Python room)
ZIP = "/content/drive/MyDrive/Circuitree/circuitree_v2__1_.zip"
W = "/content/drive/MyDrive/Circuitree"
from google.colab import drive
drive.mount('/content/drive')
!unzip -q -o "{ZIP}" -d /content/
!pip install -q gdown
!mkdir -p /content/weights
!gdown -q 1NGAJRedx040A4EeN9mALXsly87TRseRF -O /content/weights/pretrained.pth
!apt-get -qq install -y yosys > /dev/null
DATA = '/content/circuitree_v2'
!python -c "import torch, numpy; print('torch', torch.__version__, '| numpy', numpy.__version__)"
!which yosys-abc

## 2. Code files (just run them)

In [ ]:
%%writefile /content/eval_retrieval.py
"""
eval_retrieval.py  --  measure FGNN2 on circuit retrieval
Metrics: retrieval@k (hit@k), MRR, top-1 accuracy, and timing.

Put this file in FGNN2/src/ (it imports FunctionConv.py).

Expected data layout (one folder per source circuit, variants inside):
    bench_root/
        c432/   c432_v0.bench  c432_v1.bench ...   <- all variants of c432
        c880/   c880_v0.bench  ...
Two circuits "match" if they come from the same folder.

Example:
    python eval_retrieval.py --bench_root ../data/iscas_variants \
        --weights ../checkpoints/example/weight.pth --flag_inv --ks 1 5 10
"""
import argparse, os, time, json
import numpy as np


# ---------------------------------------------------------------- parsing
def parse_bench(path):
    """Parse an ABC-style AIG .bench (INPUT/OUTPUT/AND/NOT/BUFF, multi-output).
    NOT gates are folded into inverted edges (edge flag r=1), same idea as
    pretrain_dataparser.py, but this version supports many outputs."""
    inputs, outputs, defs = [], [], {}
    with open(path) as f:
        for raw in f:
            line = raw.split('#')[0].strip()
            if not line:
                continue
            up = line.upper()
            if up.startswith('INPUT('):
                inputs.append(line[line.find('(') + 1:line.rfind(')')].strip())
            elif up.startswith('OUTPUT('):
                outputs.append(line[line.find('(') + 1:line.rfind(')')].strip())
            elif '=' in line:
                lhs, rhs = [s.strip() for s in line.split('=', 1)]
                if '(' in rhs:
                    gate = rhs[:rhs.find('(')].strip().upper()
                    args = [a.strip() for a in rhs[rhs.find('(') + 1:rhs.rfind(')')].split(',') if a.strip()]
                else:                      # e.g. "n0 = vdd" / "n0 = gnd"
                    gate, args = rhs.strip().upper(), []
                defs[lhs] = (gate, args)

    def resolve(sig, depth=0):
        """Follow NOT/BUFF chains -> (real node name, inverted?)"""
        inv = False
        while sig in defs and defs[sig][0] in ('NOT', 'BUFF', 'BUF'):
            g, a = defs[sig]
            if g == 'NOT':
                inv = not inv
            sig = a[0]
        return sig, inv

    nid = {}
    def get_id(name):
        if name not in nid:
            nid[name] = len(nid)
        return nid[name]

    for pi in inputs:
        get_id(pi)
    src, dst, r = [], [], []
    for name, (gate, args) in defs.items():
        if gate == 'AND':
            d = get_id(name)
            for a in args:
                s, inv = resolve(a)
                src.append(get_id(s)); dst.append(d); r.append(int(inv))
        elif gate in ('NOT', 'BUFF', 'BUF'):
            continue
        elif gate in ('VDD', 'GND', 'CONST0', 'CONST1'):
            get_id(name)              # constant: source node, like a PI
        else:
            raise ValueError(f'{path}: gate {gate} is not AIG. Run "strash" in ABC first.')

    po_nids, po_inv = [], []
    for po in outputs:
        s, inv = resolve(po)
        po_nids.append(get_id(s)); po_inv.append(int(inv))

    return dict(n_nodes=len(nid), src=src, dst=dst, r=r,
                po_nids=po_nids, po_inv=po_inv,
                n_pi=len(inputs), n_and=sum(1 for g, _ in defs.values() if g == 'AND'))


def build_graph(p):
    import torch as th, dgl
    g = dgl.graph((th.tensor(p['src'], dtype=th.long), th.tensor(p['dst'], dtype=th.long)),
                  num_nodes=p['n_nodes'])
    g.edata['r'] = th.tensor(p['r'], dtype=th.long)
    g.ndata['inv'] = th.zeros(p['n_nodes'])      # PO inversion handled after the GNN
    # input position: input i (the i-th INPUT line, e.g. x0, x1, ...) gets i+1; every other node gets 0.
    # Only used by models that have a pos_emb table (Experiment B); ignored otherwise.
    pos = th.zeros(p['n_nodes'], dtype=th.long)
    pos[:p['n_pi']] = th.arange(1, p['n_pi'] + 1)
    g.ndata['pos'] = pos
    return g


# ---------------------------------------------------------------- embedding
def embed_batch(model, graphs, po_lists, po_inv_lists, device):
    """Embed many circuits in ONE forward pass (dgl.batch).
    Returns [num_graphs, 2*hidden] = concat(mean, max) over PO embeddings,
    the same readout train_global.py uses."""
    import torch as th, dgl
    bg = dgl.batch(graphs).to(device)
    h0 = th.ones((bg.num_nodes(), model.hidden_dim), device=device)
    pos_emb = getattr(model, 'pos_emb', None)
    if pos_emb is not None:                                   # Experiment B: each input gets its own start
        p = bg.ndata['pos']
        p = th.where(p < pos_emb.num_embeddings, p, th.zeros_like(p))
        h0 = h0 + pos_emb(p)                                  # row 0 is always zero -> gates unchanged
    bg.ndata['h'] = h0
    topo = [l.to(device) for l in dgl.topological_nodes_generator(bg)]
    use_proj = model.flag_proj
    model.set_flag_proj(False)                                # project only after PO inversion
    h = model(bg, topo, None, flag_usage='global')            # all node embeddings
    model.set_flag_proj(use_proj)

    offs = th.cumsum(th.tensor([0] + [g.num_nodes() for g in graphs[:-1]]), 0)
    po_idx, seg, inv = [], [], []
    for k, (pl, il) in enumerate(zip(po_lists, po_inv_lists)):
        po_idx += [int(offs[k]) + p for p in pl]; seg += [k] * len(pl); inv += il
    po_idx = th.tensor(po_idx, device=device); seg = th.tensor(seg, device=device)
    inv = th.tensor(inv, device=device, dtype=th.bool)

    po_h = h[po_idx].clone()
    if model.flag_inv and inv.any():
        po_h[inv] = model.func_inv(po_h[inv])                 # inverted outputs
    if use_proj:
        po_h = model.proj_head(po_h)

    B, D = len(graphs), po_h.shape[1]
    cnt = th.zeros(B, device=device).index_add_(0, seg, th.ones_like(seg, dtype=th.float))
    mean = th.zeros(B, D, device=device).index_add_(0, seg, po_h) / cnt.clamp(min=1)[:, None]
    mx = th.full((B, D), -1e9, device=device).scatter_reduce(
        0, seg[:, None].expand(-1, D), po_h, reduce='amax', include_self=True)
    return th.cat([mean, mx], 1)


# ---------------------------------------------------------------- metrics
def retrieval_metrics(emb, groups, ks=(1, 5, 10)):
    """Leave-one-out retrieval. Every circuit is a query; the rest is the gallery.
    hit@k = was at least one same-group circuit in the top k?
    MRR   = average of 1/(rank of first same-group circuit)
    acc   = top-1 nearest neighbour has the right group (== hit@1)"""
    emb = np.asarray(emb, dtype=np.float64)
    emb = emb / (np.linalg.norm(emb, axis=1, keepdims=True) + 1e-12)
    groups = np.asarray(groups)
    sim = emb @ emb.T
    np.fill_diagonal(sim, -np.inf)
    order = np.argsort(-sim, axis=1)
    rel = groups[order] == groups[:, None]
    has = rel.any(1)                                  # skip groups of size 1
    first = rel[has].argmax(1) + 1                    # 1-based rank
    out = {f'hit@{k}': float((first <= k).mean()) for k in ks}
    out['MRR'] = float((1.0 / first).mean())
    out['top1_acc'] = out.get('hit@1', float((first == 1).mean()))
    out['mean_rank'] = float(first.mean())
    out['num_queries'] = int(has.sum())
    return out


# ---------------------------------------------------------------- main
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--bench_root', required=True)
    ap.add_argument('--weights', default=None, help='pretrained FuncConv state_dict (.pth)')
    ap.add_argument('--hidden_dim', type=int, default=256)
    ap.add_argument('--out_dim', type=int, default=256)
    ap.add_argument('--flag_inv', action='store_true')
    ap.add_argument('--flag_proj', action='store_true')
    ap.add_argument('--batch_size', type=int, default=64)
    ap.add_argument('--ks', type=int, nargs='+', default=[1, 5, 10])
    ap.add_argument('--gpu', type=int, default=0)
    ap.add_argument('--out_json', default=None)
    ap.add_argument('--show_misses', action='store_true',
                    help='print every circuit whose #1 pick was not a twin')
    a = ap.parse_args()

    import torch as th
    from FunctionConv import FuncConv
    device = th.device(f'cuda:{a.gpu}' if th.cuda.is_available() else 'cpu')
    sync = (lambda: th.cuda.synchronize()) if device.type == 'cuda' else (lambda: None)

    model = FuncConv(a.hidden_dim, a.out_dim, flag_proj=a.flag_proj, flag_inv=a.flag_inv).to(device)
    if a.weights:
        model.load_state_dict(th.load(a.weights, map_location=device, weights_only=True))
    else:
        print('WARNING: no --weights, using a random untrained model (useful as a baseline)')
    model.eval()

    # 1) parse
    t0 = time.perf_counter()
    items = []
    for grp in sorted(os.listdir(a.bench_root)):
        gdir = os.path.join(a.bench_root, grp)
        if not os.path.isdir(gdir):
            continue
        for fn in sorted(os.listdir(gdir)):
            if fn.endswith('.bench'):
                items.append((grp, fn, parse_bench(os.path.join(gdir, fn))))
    t_parse = time.perf_counter() - t0
    print(f'parsed {len(items)} circuits from {len({i[0] for i in items})} groups')

    # 2) build graphs
    t0 = time.perf_counter()
    graphs = [build_graph(p) for _, _, p in items]
    t_build = time.perf_counter() - t0

    # 3) embed (timed)
    embs = []
    with th.no_grad():
        embed_batch(model, graphs[:2], [p['po_nids'] for *_, p in items[:2]],
                    [p['po_inv'] for *_, p in items[:2]], device)      # warm-up
        sync(); t0 = time.perf_counter()
        for s in range(0, len(graphs), a.batch_size):
            chunk = items[s:s + a.batch_size]
            embs.append(embed_batch(model, graphs[s:s + a.batch_size],
                                    [p['po_nids'] for *_, p in chunk],
                                    [p['po_inv'] for *_, p in chunk], device).cpu())
        sync(); t_embed = time.perf_counter() - t0
    emb = th.cat(embs).numpy()

    # 4) search + metrics (timed)
    t0 = time.perf_counter()
    res = retrieval_metrics(emb, [g for g, _, _ in items], a.ks)
    t_search = time.perf_counter() - t0

    n = len(items)
    res.update({
        'num_circuits': n, 'device': str(device),
        'time_parse_s': t_parse, 'time_build_s': t_build,
        'time_embed_s': t_embed, 'ms_per_circuit_embed': 1000 * t_embed / n,
        'time_search_all_queries_s': t_search,
        'avg_and_nodes': float(np.mean([p['n_and'] for *_, p in items])),
    })
    for k, v in res.items():
        print(f'{k:>28}: {v:.4f}' if isinstance(v, float) else f'{k:>28}: {v}')
    if a.show_misses:
        e = emb / (np.linalg.norm(emb, axis=1, keepdims=True) + 1e-12)
        sim = e @ e.T
        np.fill_diagonal(sim, -np.inf)
        top = sim.argmax(1)
        print('\nMISSES (circuit -> its #1 pick, which was NOT a twin):')
        found = False
        for i in range(n):
            if items[top[i]][0] != items[i][0]:
                found = True
                print(f'  {items[i][1]:>20} -> {items[top[i]][1]:<20} similarity {sim[i, top[i]]:.4f}')
        if not found:
            print('  none!')
    if a.out_json:
        with open(a.out_json, 'w') as f:
            json.dump(res, f, indent=2)


if __name__ == '__main__':
    main()


In [ ]:
%%writefile /content/fgnn2_torch.py
"""
fgnn2_torch.py -- FGNN2 fingerprints using ONLY PyTorch (no DGL needed).

Same math as FGNN2's FunctionConv.forward_global, rewritten without DGL so it runs anywhere
PyTorch runs (Kaggle, Windows, ...). Loads the same weight files (pretrained, Exp A, B, C).

For every topological level after the inputs, for every gate in that level:
    message from each input wire = h[source], passed through func_inv if the wire has a NOT
    neigh = average of the messages
    h[gate] = func_and(neigh)
Circuit fingerprint = concat(mean, max) over the output nodes (NOT outputs go through func_inv),
exactly like embed_circuitree.py.

Example:
    python fgnn2_torch.py --data_root DATA --weights expB_best.pth --positions --out B_val.npz
"""
import argparse, csv, os, sys, time
import numpy as np
import torch as th
from torch import nn

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from eval_retrieval import parse_bench          # pure Python, no DGL


class MLP(nn.Module):                           # identical to FunctionConv.MLP (same weight names)
    def __init__(self, *sizes, negative_slope=0.01):
        super().__init__()
        fcs = []
        for i in range(1, len(sizes)):
            fcs.append(nn.Linear(sizes[i - 1], sizes[i]))
            if i < len(sizes) - 1:
                fcs.append(nn.LeakyReLU(negative_slope=negative_slope))
        self.layers = nn.Sequential(*fcs)

    def forward(self, x):
        return self.layers(x)


class FGNN2(nn.Module):
    def __init__(self, hidden=256, positions=False, max_inputs=256):
        super().__init__()
        self.hidden = hidden
        self.func_inv = MLP(hidden, hidden // 2, hidden // 2, hidden)
        self.func_and = MLP(hidden, hidden // 2, hidden // 2, hidden)
        self.proj_head = MLP(hidden, hidden, hidden, negative_slope=0)    # unused, kept so weights load
        self.pos_emb = nn.Embedding(max_inputs + 1, hidden, padding_idx=0) if positions else None
        if positions:
            nn.init.zeros_(self.pos_emb.weight)

    def load(self, path):
        if not path:
            return
        state = th.load(path, map_location='cpu', weights_only=True)
        missing = [k for k in self.state_dict() if k not in state]
        self.load_state_dict(state, strict=not missing)
        if missing:
            print('note: these parts were not in the weight file and start at zero:', sorted({k.split('.')[0] for k in missing}))


def levels(p):
    """Topological level of every node (inputs = 0), same frontiers as dgl.topological_nodes_generator."""
    n = p['n_nodes']
    fanout = [[] for _ in range(n)]
    indeg = np.zeros(n, dtype=np.int64)
    for s, d in zip(p['src'], p['dst']):
        fanout[s].append(d); indeg[d] += 1
    lvl = np.zeros(n, dtype=np.int64)
    frontier = [i for i in range(n) if indeg[i] == 0]
    k = 0
    while frontier:
        nxt = []
        for u in frontier:
            lvl[u] = k
            for v in fanout[u]:
                indeg[v] -= 1
                if indeg[v] == 0:
                    nxt.append(v)
        frontier = nxt; k += 1
    return lvl


def embed_core(model, parsed, device='cpu', pos_lists=None):
    """Fingerprints for a list of parsed circuits, all in one batch -> tensor [len(parsed), 2*hidden].
    Works with gradients (training) or inside torch.no_grad() (scoring).
    pos_lists[k] (optional) = custom input positions for circuit k (default 1, 2, 3, ...)."""
    offs = np.cumsum([0] + [p['n_nodes'] for p in parsed])
    N = int(offs[-1])
    src = th.tensor(np.concatenate([np.asarray(p['src'], dtype=np.int64) + o for p, o in zip(parsed, offs)]), device=device)
    dst = th.tensor(np.concatenate([np.asarray(p['dst'], dtype=np.int64) + o for p, o in zip(parsed, offs)]), device=device)
    r = th.tensor(np.concatenate([np.asarray(p['r'], dtype=np.int64) for p in parsed]), device=device).bool()
    lvl = th.tensor(np.concatenate([p['lvl'] if 'lvl' in p else levels(p) for p in parsed]), device=device)

    h = th.ones(N, model.hidden, device=device)
    if model.pos_emb is not None:
        pos = th.zeros(N, dtype=th.long)
        for k, (p, o) in enumerate(zip(parsed, offs)):
            if pos_lists is not None and pos_lists[k] is not None:
                pos[o:o + p['n_pi']] = th.tensor(pos_lists[k], dtype=th.long)
            else:
                pos[o:o + p['n_pi']] = th.arange(1, p['n_pi'] + 1)
        pos = pos.clamp(max=model.pos_emb.num_embeddings - 1).to(device)
        h = h + model.pos_emb(pos)

    edge_lvl = lvl[dst]
    for k in range(1, int(lvl.max()) + 1):
        e = (edge_lvl == k).nonzero(as_tuple=True)[0]
        s, d, inv = src[e], dst[e], r[e]
        msg = h[s].clone()
        if inv.any():
            msg[inv] = model.func_inv(msg[inv])
        agg = th.zeros(N, model.hidden, device=device).index_add(0, d, msg)
        cnt = th.zeros(N, device=device).index_add(0, d, th.ones(len(d), device=device))
        nodes = (lvl == k).nonzero(as_tuple=True)[0]
        h = h.index_put((nodes,), model.func_and(agg[nodes] / cnt[nodes].clamp(min=1)[:, None]))

    out = []
    for p, o in zip(parsed, offs):
        po = th.tensor(p['po_nids'], dtype=th.long, device=device) + int(o)
        ph = h[po]
        pinv = th.tensor(p['po_inv'], dtype=th.bool, device=device)
        if pinv.any():
            ph = th.where(pinv[:, None], model.func_inv(ph), ph)
        out.append(th.cat([ph.mean(0), ph.max(0).values]))
    return th.stack(out)


@th.no_grad()
def embed(model, parsed, device='cpu', pos_lists=None):
    return embed_core(model, parsed, device, pos_lists).cpu().numpy()


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--data_root', required=True)
    ap.add_argument('--track', default='real')
    ap.add_argument('--split', default='val')
    ap.add_argument('--weights', required=True)
    ap.add_argument('--positions', action='store_true')
    ap.add_argument('--batch_size', type=int, default=256)
    ap.add_argument('--out', required=True)
    a = ap.parse_args()

    device = 'cuda' if th.cuda.is_available() else 'cpu'
    model = FGNN2(positions=a.positions)
    model.load(a.weights)
    model.to(device).eval()

    with open(os.path.join(a.data_root, 'samples.csv'), newline='') as f:
        rows = [r for r in csv.DictReader(f) if r['track'] == a.track and r['split'] == a.split]
    rows.sort(key=lambda r: int(r['depth']))
    t = time.perf_counter()
    parsed = [parse_bench(os.path.join(a.data_root, r['file'])) for r in rows]
    print(f'{len(rows)} circuits read in {time.perf_counter() - t:.0f} s')

    t = time.perf_counter()
    embs = []
    for s in range(0, len(parsed), a.batch_size):
        embs.append(embed(model, parsed[s:s + a.batch_size], device))
    dt = time.perf_counter() - t
    np.savez(a.out, sample_ids=np.array([r['sample_id'] for r in rows]), embeddings=np.concatenate(embs))
    print(f'saved {a.out}  (embedding took {dt:.0f} s, {1000 * dt / len(rows):.1f} ms per circuit on CPU)')


if __name__ == '__main__':
    main()


In [ ]:
%%writefile /content/strict_eval.py
"""
strict_eval.py -- "strict" grader: swapped-input copies count as WRONG answers.

The official evaluate.py removes swapped-input copies (same perm_group_id, different function)
before scoring. This grader keeps them in, so its full-search scores match exactly what
top20_cec.py shows. Ranking rule (shared with top20_cec.py): highest cosine similarity first,
ties broken by sample_id.

Hard lineups contain no swapped-input copies, so strict hard = official hard.
Only needs numpy.

Example:
    python strict_eval.py --data_root /content/circuitree_v2 --emb /content/expB_real_val.npz
    python strict_eval.py ... --query r029642     # also show one query, to compare with its top-20 table
"""
import argparse, csv, os, sys
import numpy as np

KS = (1, 5, 10, 20)


def add_input_count(E, num_inputs, weight=10.0, max_inputs=256):
    """Input-count rule (same as fgnn2_pos.add_input_count): circuits with a different number of
    inputs can never look close."""
    E = E / (np.linalg.norm(E, axis=1, keepdims=True) + 1e-12)
    oh = np.zeros((len(E), max_inputs + 1))
    oh[np.arange(len(E)), np.minimum(np.asarray(num_inputs), max_inputs)] = weight
    return np.hstack([E, oh])


def rank(sim, ids_arr):
    """Order used by both strict_eval.py and top20_cec.py: highest similarity first, ties by sample_id."""
    return np.lexsort((ids_arr, -sim))


def query_metrics(is_twin_ranked):
    pos = np.flatnonzero(is_twin_ranked)
    if len(pos) == 0:
        return None
    m = {'hit1': float(is_twin_ranked[0]), 'mrr': 1.0 / (pos[0] + 1)}
    for k in KS[1:]:
        m[f'r{k}'] = is_twin_ranked[:k].sum() / len(pos)          # share of all twins in the top k
    m['any20'] = float(pos[0] < 20)                                 # at least one twin in the top 20
    m['ap'] = float((np.arange(1, len(pos) + 1) / (pos + 1)).mean())
    return m


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--data_root', required=True)
    ap.add_argument('--emb', required=True)
    ap.add_argument('--track', default='real')
    ap.add_argument('--split', default='val')
    ap.add_argument('--query', default=None, help='also print this one query (to check against top20_cec.py)')
    ap.add_argument('--input_count', action='store_true', help='apply the input-count rule first')
    a = ap.parse_args()

    sys.path.insert(0, a.data_root)
    import evaluate as ev

    with open(os.path.join(a.data_root, 'samples.csv'), newline='') as f:
        info = {r['sample_id']: r for r in csv.DictReader(f)}
    z = np.load(a.emb)
    ids = [str(s) for s in z['sample_ids']]
    ids_arr = np.array(ids)
    E = z['embeddings'].astype(np.float64)
    if a.input_count:
        E = add_input_count(E, [int(info[s]['num_inputs']) for s in ids])
    E /= np.linalg.norm(E, axis=1, keepdims=True) + 1e-12
    fn = np.array([info[s]['function_id'] for s in ids])
    idx = {s: i for i, s in enumerate(ids)}
    with open(os.path.join(a.data_root, 'eval', f'{a.track}_{a.split}_queries.csv'), newline='') as f:
        queries = [r['query_id'] for r in csv.DictReader(f)]

    res = []
    for q in queries:
        i = idx[q]
        sim = E @ E[i]
        order = rank(sim, ids_arr)
        order = order[order != i]                                   # not itself
        m = query_metrics(fn[order] == fn[i])
        if m:
            res.append(m)
            if q == a.query:
                print(f"query {q}: Hit@1 {m['hit1']:.0f} | MRR {m['mrr']:.3f} | "
                      f"twins in top 10 {m['r10']:.2f} | top 20 {m['r20']:.2f}\n")
    strict = {k: float(np.mean([r[k] for r in res])) for k in res[0]}

    emb = dict(zip(ids, E))
    off_full = ev.evaluate(emb, a.track, a.split, 'full', root=a.data_root)
    off_hard = ev.evaluate(emb, a.track, a.split, 'hard', root=a.data_root)

    print(f"{a.track} {a.split} | {os.path.basename(a.emb)}{' + input-count rule' if a.input_count else ''} | "
          f"{len(res)} full-search queries")
    print(f"{'':28}{'Hit@1':>7}{'MRR':>7}{'top5':>7}{'top10':>7}{'top20':>7}{'>=1 in 20':>11}{'AP':>7}")
    print(f"{'full, STRICT (copies wrong)':28}{strict['hit1']:7.3f}{strict['mrr']:7.3f}{strict['r5']:7.3f}"
          f"{strict['r10']:7.3f}{strict['r20']:7.3f}{strict['any20']:11.3f}{strict['ap']:7.3f}")
    print(f"{'full, official (copies out)':28}{off_full['hit1']:7.3f}{off_full['mrr']:7.3f}{off_full['r5']:7.3f}"
          f"{off_full['r10']:7.3f}{'-':>7}{'-':>11}{off_full['ap']:7.3f}")
    print(f"{'hard (same either way)':28}{off_hard['hit1']:7.3f}{off_hard['mrr']:7.3f}{off_hard['r5']:7.3f}"
          f"{off_hard['r10']:7.3f}{'-':>7}{'-':>11}{off_hard['ap']:7.3f}")
    print('\ntopK = share of ALL of a query\'s twins that land in the top K (averaged over queries)')


if __name__ == '__main__':
    main()


In [ ]:
%%writefile /content/top20_cec.py
"""
top20_cec.py -- for one query circuit, show the top-k most similar circuits by FGNN2
fingerprint, and check each one with ABC's CEC (combinational equivalence checking).

Needs: the .npz from embed_circuitree.py, the circuitree_v2 folder, and yosys-abc
(Colab: !apt-get -qq install -y yosys). Only numpy is needed, no PyTorch.

Example:
    python top20_cec.py --data_root /content/circuitree_v2 --emb /content/fgnn2_real_val.npz \
        --track real --split val --query r027344 --k 20 --out /content/top20_r027344.csv
Leave out --query to pick a random evaluation query (use --seed to change which one).
"""
import argparse, csv, os, random, subprocess, time
import numpy as np


def add_input_count(E, num_inputs, weight=10.0, max_inputs=256):
    """Input-count rule (same as fgnn2_pos.add_input_count): circuits with a different number of
    inputs can never look close."""
    E = E / (np.linalg.norm(E, axis=1, keepdims=True) + 1e-12)
    oh = np.zeros((len(E), max_inputs + 1))
    oh[np.arange(len(E)), np.minimum(np.asarray(num_inputs), max_inputs)] = weight
    return np.hstack([E, oh])


def run_cec(abc, a, b):
    """Return (result, seconds). result: EQUIVALENT / NOT EQUIVALENT / DIFFERENT #INPUTS / ERROR"""
    t = time.perf_counter()
    out = subprocess.run([abc, '-q', f'cec {a} {b}'], capture_output=True, text=True, timeout=120)
    sec = time.perf_counter() - t
    text = out.stdout + out.stderr
    if 'NOT EQUIVALENT' in text:
        return 'NOT EQUIVALENT', sec
    if 'are equivalent' in text:
        return 'EQUIVALENT', sec
    if 'different number of primary inputs' in text:
        return 'DIFFERENT #INPUTS', sec
    return 'ERROR', sec


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--data_root', required=True)
    ap.add_argument('--emb', required=True, help='.npz from embed_circuitree.py')
    ap.add_argument('--track', default='real')
    ap.add_argument('--split', default='val')
    ap.add_argument('--query', default=None, help='sample_id; leave out for a random query')
    ap.add_argument('--seed', type=int, default=0)
    ap.add_argument('--k', type=int, default=20)
    ap.add_argument('--abc', default='yosys-abc')
    ap.add_argument('--out', default=None, help='optional .csv to save the table')
    ap.add_argument('--input_count', action='store_true', help='apply the input-count rule first')
    a = ap.parse_args()

    with open(os.path.join(a.data_root, 'samples.csv'), newline='') as f:
        info = {r['sample_id']: r for r in csv.DictReader(f)}

    z = np.load(a.emb)
    ids = z['sample_ids'].tolist()
    emb = z['embeddings'].astype(np.float64)
    if a.input_count:
        emb = add_input_count(emb, [int(info[s]['num_inputs']) for s in ids])
    emb /= np.linalg.norm(emb, axis=1, keepdims=True) + 1e-12

    q = a.query
    if q is None:
        qfile = os.path.join(a.data_root, 'eval', f'{a.track}_{a.split}_queries.csv')
        with open(qfile, newline='') as f:
            q = random.Random(a.seed).choice([r['query_id'] for r in csv.DictReader(f)])
    qi = ids.index(q)
    qrow = info[q]

    sim = emb @ emb[qi]
    # same ranking rule as strict_eval.py: highest similarity first, ties broken by sample_id
    order = np.lexsort((np.array(ids), -sim))
    top = order[order != qi][:a.k]                      # don't return the query itself

    total_twins = sum(1 for s in ids if s != q and info[s]['function_id'] == qrow['function_id'])
    qpath = os.path.join(a.data_root, qrow['file'])
    print(f"QUERY {q}: {qrow['num_inputs']} inputs, {qrow['and_count']} ANDs, depth {qrow['depth']}, "
          f"from {qrow['source_circuit']}")
    print(f"gallery = all {len(ids) - 1} other circuits in {a.track} {a.split}; "
          f"true twins in gallery: {total_twins}\n")

    header = ['rank', 'sample_id', 'similarity', 'inputs', 'ANDs', 'depth', 'role', 'label_twin', 'CEC', 'cec_s']
    print(f"{'rank':>4}  {'sample_id':<9} {'sim':>7} {'in':>4} {'ANDs':>5} {'dep':>4}  {'role':<9} "
          f"{'twin?':<5}  CEC")
    table, n_eq, agree, t_cec = [], 0, 0, 0.0
    for rank, j in enumerate(top, 1):
        s = ids[j]; r = info[s]
        label = 'yes' if r['function_id'] == qrow['function_id'] else 'no'
        cec, sec = run_cec(a.abc, qpath, os.path.join(a.data_root, r['file']))
        t_cec += sec
        n_eq += cec == 'EQUIVALENT'
        agree += (cec == 'EQUIVALENT') == (label == 'yes')
        table.append([rank, s, f'{sim[j]:.4f}', r['num_inputs'], r['and_count'], r['depth'],
                      r['role'], label, cec, f'{sec:.3f}'])
        print(f"{rank:>4}  {s:<9} {sim[j]:>7.4f} {r['num_inputs']:>4} {r['and_count']:>5} {r['depth']:>4}  "
              f"{r['role']:<9} {label:<5}  {cec}")

    twin_ranks = [row[0] for row in table if row[7] == 'yes']
    strict_mrr = f'{1 / twin_ranks[0]:.3f}' if twin_ranks else f'< {1 / (a.k + 1):.3f}'
    print(f"\nthis query, strict scores (copies count as wrong): Hit@1 {int(bool(twin_ranks) and twin_ranks[0] == 1)} | "
          f"MRR {strict_mrr} | twins in top {a.k}: {len(twin_ranks)}/{total_twins}")
    print(f"CEC says {n_eq} of the top {a.k} are truly equivalent "
          f"(out of {total_twins} twins that exist).")
    print(f"CEC agrees with the dataset labels on {agree}/{a.k}.  Total CEC time: {t_cec:.2f} s")
    if a.out:
        with open(a.out, 'w', newline='') as f:
            w = csv.writer(f); w.writerow(header); w.writerows(table)
        print(f'saved {a.out}')


if __name__ == '__main__':
    main()


## 3. Strict scores (swapped-input copies count as wrong)

In [ ]:
# 3a: fingerprints for pretrained, Exp A, Exp B (a few minutes each on CPU)
!python /content/fgnn2_torch.py --data_root {DATA} --weights /content/weights/pretrained.pth --out /content/pre_val.npz
!python /content/fgnn2_torch.py --data_root {DATA} --weights {W}/expA/best.pth --out /content/A_val.npz
!python /content/fgnn2_torch.py --data_root {DATA} --weights {W}/expB/best.pth --positions --out /content/B_val.npz

In [ ]:
# 3b: correctness check first (no rule), then all 4 models WITH the input-count rule
for m in ['pre', 'B']:
    !python /content/strict_eval.py --data_root {DATA} --emb /content/{m}_val.npz
    print()
print('=' * 30, 'WITH input-count rule', '=' * 30)
for m in ['pre', 'A', 'B']:   # B + --input_count = Exp C (final)
    !python /content/strict_eval.py --data_root {DATA} --emb /content/{m}_val.npz --input_count
    print()

In [ ]:
# 3c: table and scores agree -- one query, same model, same rule
!python /content/strict_eval.py --data_root {DATA} --emb /content/B_val.npz --input_count --query r029642
!python /content/top20_cec.py --data_root {DATA} --emb /content/B_val.npz --input_count --query r029642 --k 20 --out {W}/expB/top20_B_strict.csv